# Синтаксис: чанкинг, грамматики и парсинг



## Русский язык: чем размечать

**Главное различие:** морфологический анализатор перечисляет разборы отдельного слова;
контекстный теггер выбирает метку в предложении. POS — часть речи;
падеж, число и род — отдельные признаки. Модели тоже ошибаются.

| Инструмент | Что даёт | Особенность |
|---|---|---|
| [pymorphy3](https://pymorphy3.readthedocs.io/en/latest/user/guide.html) | Леммы и морфологические разборы | Без контекста; схема OpenCorpora; `parse()[0]` не учитывает предложение |
| [Natasha](https://github.com/natasha/natasha) | Контекстные POS и признаки; отдельно — синтаксис и NER | Русские модели, ориентированные на новости |
| [spaCy](https://spacy.io/models/ru) | POS, признаки, леммы, зависимости, NER | Нужен русский пакет модели; удобно получать поля каждого токена |
| [Stanza](https://stanfordnlp.github.io/stanza/pos.html) | Контекстные POS, признаки, леммы; отдельно — зависимости | Схема UD; модели скачиваются отдельно от библиотеки |
| [NLTK](https://www.nltk.org/api/nltk.tag.html) | Контекстная русская POS-разметка | `lang='rus'`, отдельная модель; по умолчанию теги НКРЯ, не Penn |
| [pymystem3 / MyStem](https://pypi.org/project/pymystem3/) | Контекстный морфологический анализ и леммы | Python-обёртка над внешней программой MyStem; своя схема тегов |

**Проверочный пример:** «Они **стали** читать» / «Деталь сделана из **стали**».
Ожидаем глагол с леммой «стать» и существительное с леммой «сталь».

Ниже — короткие примеры-справка. Каждый запускается отдельно после установки указанного инструмента.
Для основного английского маршрута занятия устанавливать их все не требуется.


### pymorphy3 — возможные разборы слова

Установка: `pip install pymorphy3`. Выводим **все варианты**, а не выбираем первый автоматически.

```python
import pymorphy3
morph = pymorphy3.MorphAnalyzer()
for variant in morph.parse('стали'):
    print(variant.normal_form, variant.tag.POS, variant.tag.case)
```


### Natasha — морфология в контексте

Установка: `pip install natasha`. Сначала сегментация, затем контекстная разметка.

```python
from natasha import Doc, Segmenter, NewsEmbedding, NewsMorphTagger
doc = Doc('Они стали читать. Деталь сделана из стали.')
doc.segment(Segmenter())
doc.tag_morph(NewsMorphTagger(NewsEmbedding()))
for token in doc.tokens:
    if token.text == 'стали':
        print(token.text, token.pos, token.feats)
```


### spaCy — единый конвейер

Установка: `pip install spacy`; модель: `python -m spacy download ru_core_news_sm`.
`pos_` — часть речи, `morph` — признаки, `lemma_` — лемма.

```python
import spacy
nlp = spacy.load('ru_core_news_sm')
doc = nlp('Они стали читать. Деталь сделана из стали.')
for token in doc:
    if token.text == 'стали':
        print(token.text, token.pos_, token.morph, token.lemma_)
```


### Stanza — разметка в схеме UD

Установка: `pip install stanza`. При первом запуске загружаются выбранные модели;
`upos` — часть речи, `feats` — грамматические признаки.

```python
import stanza
nlp = stanza.Pipeline('ru', processors='tokenize,pos,lemma', use_gpu=False)
doc = nlp('Они стали читать. Деталь сделана из стали.')
for sentence in doc.sentences:
    for word in sentence.words:
        if word.text == 'стали':
            print(word.text, word.upos, word.feats, word.lemma)
```


### NLTK — русский теггер тоже есть

Установка: `pip install nltk`. Передаём токены **каждого предложения отдельно**;
английская модель для этого не подходит. Здесь получаем POS, не полный морфологический разбор.

```python
import nltk
nltk.download('averaged_perceptron_tagger_rus', raise_on_error=True)
sentences = [['Они', 'стали', 'читать', '.'],
             ['Деталь', 'сделана', 'из', 'стали', '.']]
for tokens in sentences:
    print(nltk.pos_tag(tokens, lang='rus'))
```


### pymystem3 — доступ к MyStem

Установка: `pip install pymystem3`. Требуется внешний бинарник MyStem;
обёртка пытается загрузить его при первом запуске. `analysis` содержит лемму и грамматическую характеристику.

```python
from pymystem3 import Mystem
mystem = Mystem()
for token in mystem.analyze('Они стали читать. Деталь сделана из стали.'):
    if token['text'] == 'стали':
        print(token['text'], token.get('analysis', []))
```




## Подготовка

Нужны NLTK и его данные. При первом запуске ресурсы скачиваются из интернета;
при повторном используются уже загруженные. Большие модели других библиотек не нужны.
Деревья будем печатать прямо в ноутбуке через `pretty_print()`, без отдельного графического окна.


In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('nltk') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nltk>=3.9,<4'])

import nltk
from collections import Counter, defaultdict
from nltk import CFG, Tree
from nltk.corpus import brown, wordnet as wn, conll2000, gutenberg
from nltk.chunk import tree2conlltags, conlltags2tree, ChunkScore
from nltk.tokenize import word_tokenize, sent_tokenize

resources = {
    'punkt_tab': 'tokenizers/punkt_tab/english/',
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'universal_tagset': 'taggers/universal_tagset/',
    'wordnet': 'corpora/wordnet.zip',
    'brown': 'corpora/brown/',
    'conll2000': 'corpora/conll2000/',
    'gutenberg': 'corpora/gutenberg/',
    'maxent_ne_chunker_tab': 'chunkers/maxent_ne_chunker_tab/english_ace_multiclass/',
    'words': 'corpora/words/',
}
for package, path in resources.items():
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(package, raise_on_error=True)

print('NLTK:', nltk.__version__)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to /root/nltk_d

NLTK: 3.9.1


## 1. Зачем синтаксис, если у нас уже есть POS?

«Студенты читают книги» и «Книги читают студенты».


- Именная группа NP (noun phrase)

Красное яблоко (прилагательное + существительное).

Это красное яблоко (местоимение + прил. + сущ.).

Дом моего отца.

- Глагольная группа VP (verb phrase)

Читать книгу (глагол + объект).

Быстро читать книгу (наречие + глагол + объект).

- Предложная PP (prepositional phrase)

В комнате (предлог + сущ.).

На столе


In [3]:
tagged_example = [
    ('The', 'DT'), ('curious', 'JJ'), ('students', 'NNS'),
    ('read', 'VBP'), ('the', 'DT'), ('old', 'JJ'), ('book', 'NN'),
    ('in', 'IN'), ('the', 'DT'), ('library', 'NN'), ('.', '.'),
]
for word, pos in tagged_example:
    print(f'{word:10} {pos}')


The        DT
curious    JJ
students   NNS
read       VBP
the        DT
old        JJ
book       NN
in         IN
the        DT
library    NN
.          .


Здесь POS-разметка **задана вручную**


## 2. Чанкинг: собираем именные группы

**Чанкинг** — поверхностное выделение групп, например базовых NP, по **правилам**

Правило ниже: необязательный определитель, затем прилагательные, затем одно или несколько существительных.
Это регулярное выражение **над POS-тегами**

| Запись | Значение |
|---|---|
| `<DT>?` | ноль или один определитель |
| `<JJ.*>*` | ноль или больше тегов, начинающихся с JJ |
| `<NN.*>+` | один или больше тегов, начинающихся с NN: NN, NNS, NNP, NNPS |
| `NP: {...}` | объединить найденный фрагмент в группу с меткой NP |


### Стоп 3

Где будут границы NP в `The curious students read the old book in the library`?
Войдёт ли `in` в группу? Останутся ли за пределами NP какие-либо токены?

**До запуска следующей ячейки:** дайте 15–20 секунд подумать, затем соберите 2–3 версии в чате или голосом. Попросите объяснить причину ответа.


In [4]:
NP_GRAMMAR = r'NP: {<DT>?<JJ.*>*<NN.*>+}'
np_parser = nltk.RegexpParser(NP_GRAMMAR)
chunk_tree = np_parser.parse(tagged_example)

def show_chunk_tree(tree):
    """Печатает чанки с отдельным уровнем POS, сохраняя порядок слов."""
    picture = tree.copy(deep=True)
    for position in tree.treepositions('leaves'):
        word, pos = tree[position]
        picture[position] = Tree(pos, [word])
    picture.pretty_print()

print(chunk_tree.pformat(margin=100))
show_chunk_tree(chunk_tree)


(S
  (NP The/DT curious/JJ students/NNS)
  read/VBP
  (NP the/DT old/JJ book/NN)
  in/IN
  (NP the/DT library/NN)
  ./.)
                               S                              
        _______________________|____________________________   
       NP             |        NP       |       NP          | 
  _____|_______       |     ___|___     |    ___|_____      |  
 DT    JJ     NNS    VBP   DT  JJ  NN   IN  DT        NN    . 
 |     |       |      |    |   |   |    |   |         |     |  
The curious students read the old book  in the     library  . 



<details>
<summary>После обсуждения: пояснение</summary>

Получаются `[The curious students]`, `[the old book]`, `[the library]`.
Глагол, предлог и точка остаются вне NP. Метка `S` здесь — контейнер результата,
а не доказательство, что чанкер построил полный синтаксический разбор предложения.

</details>


In [ ]:
def noun_phrases(tree):
    """Возвращает строки NP из плоского дерева чанкинга."""
    return [' '.join(word for word, _ in child.leaves())
            for child in tree if isinstance(child, Tree) and child.label() == 'NP']

print(noun_phrases(chunk_tree))


['The curious students', 'the old book', 'the library']


### Когда понятное правило оказывается слишком узким

Добавим наречие: `a very old book`. В исходном шаблоне нет `<RB>`, поэтому наречие разрывает совпадение.


### Стоп 4

Что выделит старое правило: всю группу, только `old book` или ничего?
Как добавить наречие, не превращая грамматику в «бери любые слова подряд»?



In [ ]:
with_adverb = [('a', 'DT'), ('very', 'RB'), ('old', 'JJ'), ('book', 'NN')]
np_parser_with_adverbs = nltk.RegexpParser(r'NP: {<DT>?<RB.*>*<JJ.*>*<NN.*>+}')
print('Исходное правило:', noun_phrases(np_parser.parse(with_adverb)))
print('С наречиями:     ', noun_phrases(np_parser_with_adverbs.parse(with_adverb)))


Исходное правило: ['old book']
С наречиями:      ['a very old book']


<details>
<summary>После обсуждения: пояснение</summary>

Первое правило находит только `old book`, второе — `a very old book`.
Это улучшение **на данном примере**, а не доказательство общего превосходства.
Обновлённый шаблон всё ещё не описывает координацию, все варианты наречий и вложенность.
Нужны и положительные, и отрицательные примеры, затем отложенная проверка.

</details>


## 3. IOB: представляем границы групп метками

- `B-NP` — начало новой NP.
- `I-NP` — продолжение той же NP.
- `O` — токен вне выделяемых групп; это буква O, не ноль.

Для нашего плоского чанкинга каждый токен получает одну такую метку.
Одно IOB-представление не кодирует произвольные вложенные группы.
POS и IOB — **разные слои**: `NN` описывает часть речи, `B-NP` — границу группы.


In [ ]:
iob_example = tree2conlltags(chunk_tree)
print(f'{"Токен":10} {"POS":5} IOB')
for word, pos, iob in iob_example:
    print(f'{word:10} {pos:5} {iob}')

restored_tree = conlltags2tree(iob_example)
print('NP после обратного преобразования:', noun_phrases(restored_tree))


Токен      POS   IOB
The        DT    B-NP
curious    JJ    I-NP
students   NNS   I-NP
read       VBP   O
the        DT    B-NP
old        JJ    I-NP
book       NN    I-NP
in         IN    O
the        DT    B-NP
library    NN    I-NP
.          .     O
NP после обратного преобразования: ['The curious students', 'the old book', 'the library']


`B-NP` требуется и тогда, когда новая NP стоит сразу после другой NP того же типа.
Два `B-NP` подряд не равны `B-NP I-NP`. Проверим на **условном списке двух названий**, а не на полном предложении.


In [ ]:
separate = [('Moscow', 'NNP', 'B-NP'), ('London', 'NNP', 'B-NP')]
joined = [('Moscow', 'NNP', 'B-NP'), ('London', 'NNP', 'I-NP')]
print('Две группы:', noun_phrases(conlltags2tree(separate)))
print('Одна группа:', noun_phrases(conlltags2tree(joined)))


Две группы: ['Moscow', 'London']
Одна группа: ['Moscow London']


### Как оценивать чанкер

Токенная IOB accuracy считает совпавшие метки токенов.
Для извлечения группы часто важны **обе её границы и тип целиком**.

При точном совпадении чанков:

- Precision — сколько предсказанных групп совпало с эталоном.
- Recall — сколько эталонных групп нашли.
- F1 — гармоническое среднее precision и recall.

Эталон здесь означает принятую разметку задачи, а не «единственную истину о языке».


### Стоп 5

Эталон: `[The old book] fell .`, а модель выделила только `[The old] book fell .`,
оставив `book` вне группы. Большинство IOB-меток совпало.
Нашла ли модель хотя бы одну **целую правильную NP**?

**До запуска следующей ячейки:** дайте 15–20 секунд подумать, затем соберите 2–3 версии в чате или голосом. Попросите объяснить причину ответа.


In [ ]:
gold_boundary = conlltags2tree([
    ('The', 'DT', 'B-NP'), ('old', 'JJ', 'I-NP'), ('book', 'NN', 'I-NP'),
    ('fell', 'VBD', 'O'), ('.', '.', 'O'),
])
wrong_boundary = conlltags2tree([
    ('The', 'DT', 'B-NP'), ('old', 'JJ', 'I-NP'), ('book', 'NN', 'O'),
    ('fell', 'VBD', 'O'), ('.', '.', 'O'),
])
boundary_score = ChunkScore()
boundary_score.score(gold_boundary, wrong_boundary)
print(f'IOB accuracy: {boundary_score.accuracy():.0%}')
print(f'Chunk precision: {boundary_score.precision():.0%}')
print(f'Chunk recall:    {boundary_score.recall():.0%}')
print(f'Chunk F1:        {boundary_score.f_measure():.0%}')


IOB accuracy: 80%
Chunk precision: 0%
Chunk recall:    0%
Chunk F1:        0%


<details>
<summary>После обсуждения: пояснение</summary>

IOB accuracy — 80%, а точных совпадений NP нет: precision, recall и F1 равны нулю.
Модель угадала почти все токенные метки, но ошиблась в границе единственной группы.
Вопрос «хорошее ли качество?» требует сначала назвать единицу оценки.

</details>


### Дополнительно: правило на отложенной части CoNLL-2000

В CoNLL-2000 есть POS и эталонные чанки. Возьмём только NP.
Официальные train и test хранятся отдельно; ограничим объём для быстрого учебного запуска.
Проверка использует **эталонный POS**, поэтому оценивает чанкер при известных частях речи,
а не весь конвейер с автоматическим теггером.

Правило не будем подгонять по этим тестовым предложениям. Для подбора нужна отдельная validation-часть train.


In [ ]:
np_train = list(conll2000.chunked_sents('train.txt', chunk_types=['NP']))[:1000]
np_test = list(conll2000.chunked_sents('test.txt', chunk_types=['NP']))[:200]

def evaluate_chunker(parser, references):
    """Оценивает плоские чанки по точным границам и меткам на эталонном POS."""
    score = ChunkScore() # оценивает качество
    for reference in references:
        score.score(reference, parser.parse(reference.leaves()))
    return score

rule_np_score = evaluate_chunker(np_parser, np_test) # внутри есть метрики
print('Предложения train / test:', len(np_train), '/', len(np_test))
print(f'Правило: IOB accuracy={rule_np_score.accuracy():.2%}, '
      f'P={rule_np_score.precision():.2%}, R={rule_np_score.recall():.2%}, '
      f'F1={rule_np_score.f_measure():.2%}')


Предложения train / test: 1000 / 200
Правило: IOB accuracy=77.05%, P=66.97%, R=57.08%, F1=61.63%


### Дополнительно: обучаемый чанкер

На прошлом занятии предсказывали POS по слову. Теперь предсказываем **IOB по POS**.
Это намеренно слабый baseline: он не использует само слово или историю IOB.
Конкретные соответствия `POS → IOB` выучены из train, а не записаны вручную.


In [ ]:
class UnigramNPChunker(nltk.ChunkParserI):
    """Учебный baseline: предсказывает IOB-метку только по текущему POS."""

    def __init__(self, train_trees):
        training = [[(pos, iob) for _, pos, iob in tree2conlltags(tree)]
                    for tree in train_trees]
        self.tagger = UnigramTagger(training, backoff=DefaultTagger('O'))

    def parse(self, tagged_tokens):
        pos_tags = [pos for _, pos in tagged_tokens]
        predicted = self.tagger.tag(pos_tags)
        triples = [(word, pos, iob)
                   for (word, pos), (_, iob) in zip(tagged_tokens, predicted)]
        return conlltags2tree(triples, strict=False)

learned_np = UnigramNPChunker(np_train) # берем этот алгоритм и обучаем
learned_np_score = evaluate_chunker(learned_np, np_test)
for name, score in [('Правило', rule_np_score), ('Обучаемый baseline', learned_np_score)]:
    print(f'{name:22} IOB={score.accuracy():.2%} P={score.precision():.2%} '
          f'R={score.recall():.2%} F1={score.f_measure():.2%}')


Правило                IOB=77.05% P=66.97% R=57.08% F1=61.63%
Обучаемый baseline     IOB=91.97% P=77.99% R=86.24% F1=81.91%


Униграммная модель может выдать `I-NP` после `O` или в начале предложения.
`conlltags2tree(..., strict=False)` интерпретирует такое продолжение как начало группы.
Мы оцениваем **получившееся дерево после этого преобразования**, а не гарантируем корректность сырой IOB-последовательности.


### Стоп 6 · дополнительно

Победил ли обучаемый чанкер по F1? А по токенной accuracy?
Почему модель, которая «училась», может проиграть короткому правилу?
Каких признаков ей не хватает и как проверить ваше предложение без подгонки по test?



<details>
<summary>После обсуждения: пояснение</summary>

Сначала читаем фактические числа своего запуска. Увеличение сложности и наличие обучения не гарантируют победы.
Можно предложить слова, соседние POS, окончания или историю IOB — и проверить изменение на validation,
оставив итоговый test для финального сравнения. Один небольшой тест не даёт общего рейтинга методов.

</details>


## 4. Контекстно-свободная грамматика: строим полное дерево

Чанкер нашёл отдельные группы. Теперь хотим связать их в структуру предложения.
**Грамматика** задаёт допустимые правила, **парсер** ищет деревья, согласованные с правилами и входом.

В CFG (context-free grammar) слева у правила один нетерминальный символ.
Например, `S -> NP VP`: предложение состоит из NP и VP.
Это формальный смысл «контекстно-свободная»: правило не требует соседних символов слева.
Это не означает, что человеческий язык или значение слов вообще не зависят от контекста.

`S`, `NP`, `VP`, `Det`, `N`, `V` — категории; строки в кавычках — **терминалы**, то есть входные слова.
Здесь мы вручную описываем маленький фрагмент языка; не обучаем полную грамматику английского.


In [ ]:
simple_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N | Det Adj N | 'Alice'
    VP -> V NP
    Det -> 'the' | 'a'
    Adj -> 'old' | 'curious'
    N -> 'student' | 'book'
    V -> 'reads'
""")
print(simple_grammar)


Grammar with 12 productions (start state = S)
    S -> NP VP
    NP -> Det N
    NP -> Det Adj N
    NP -> 'Alice'
    VP -> V NP
    Det -> 'the'
    Det -> 'a'
    Adj -> 'old'
    Adj -> 'curious'
    N -> 'student'
    N -> 'book'
    V -> 'reads'


### Стоп 7

Какие узлы понадобятся для `the student reads the old book`?
Чем его полное дерево должно отличаться от плоского выделения
`[the student]` и `[the old book]` чанкером?



In [ ]:
from nltk.parse import ChartParser

simple_parser = ChartParser(simple_grammar) # парсер с разными вариантами, сохраняет все гипотезы в виде деревьев
syntax_tokens = 'the student reads the old book'.split()
simple_trees = list(simple_parser.parse(syntax_tokens))
print('Деревьев:', len(simple_trees))
for tree in simple_trees:
    tree.pretty_print()


Деревьев: 1
                  S                   
      ____________|________            
     |                     VP         
     |             ________|___        
     NP           |            NP     
  ___|_____       |     _______|___    
Det        N      V   Det     Adj  N  
 |         |      |    |       |   |   
the     student reads the     old book



<details>
<summary>После обсуждения: пояснение</summary>

Теперь есть узел VP, объединяющий глагол и объектную NP, и общий S.
Это вложенная структура. В CFG-примере вход — **слова**, а в RegexpParser входом были **пары слово/POS**.
Обозначения `NP` совпадают, но задачи и устройство правил различаются.

</details>


### «Не разобралось» не значит «неправильно по-английски»

Есть как минимум две разные причины: в лексиконе нет слова или правила не позволяют собрать структуру.
Для естественного языка это может быть просто недостаток нашей маленькой грамматики.


In [ ]:
def try_parse(parser, tokens):
    """Отделяет отсутствие слов в грамматике от отсутствия полного разбора."""
    try:
        parser.grammar().check_coverage(tokens)
    except ValueError:
        print('Нет покрытия словарём грамматики:', ' '.join(tokens))
        return []
    trees = list(parser.parse(tokens))
    print('Полных разборов:', len(trees), '|', ' '.join(tokens))
    return trees

try_parse(simple_parser, 'Alice reads the manuscript'.split())
try_parse(simple_parser, 'the student the book reads'.split())
odd_but_licensed = try_parse(simple_parser, 'the book reads the student'.split())


Нет покрытия словарём грамматики: Alice reads the manuscript
Полных разборов: 0 | the student the book reads
Полных разборов: 1 | the book reads the student


`manuscript` отсутствует в словаре. Во второй строке все слова известны, но нужной структуры нет.
Третью строку грамматика допускает, хотя буквальная ситуация необычна:
правила не кодируют знание о том, кто обычно умеет читать.
Разбор по CFG — ещё не проверка правдоподобия или фактической истинности.


### Дополнительно: одна грамматика, три парсера

| Парсер | Идея | Ограничение учебной реализации |
|---|---|---|
| Recursive descent | сверху вниз: от S к словам | наивная рекурсия опасна при левой рекурсии; может повторять работу |
| Shift-reduce | снизу вверх: сдвиг слов в стек и свёртки | NLTK-реализация жадная, без полного перебора альтернатив |
| Chart | сохраняет гипотезы о фрагментах и комбинирует их | число вариантов разбора всё равно может быть большим |

На маленькой грамматике без левой рекурсии сравним **число найденных деревьев**,
не будем объявлять один алгоритм универсально самым быстрым по одному запуску.


In [ ]:
from nltk.parse import RecursiveDescentParser, ShiftReduceParser

for parser_type in [RecursiveDescentParser, ShiftReduceParser, ChartParser]:
    parser = parser_type(simple_grammar)
    trees = list(parser.parse(syntax_tokens))
    print(f'{parser_type.__name__:24} деревьев: {len(trees)}')


RecursiveDescentParser   деревьев: 1
ShiftReduceParser        деревьев: 1
ChartParser              деревьев: 1


## 5. Синтаксическая неоднозначность

`I saw the student with a telescope.`

В одном прочтении я использовал телескоп; в другом у студента был телескоп.
В обоих случаях `with a telescope` — PP, но **прикрепляется к разным узлам**.


### Стоп 8

К чему можно присоединить `with a telescope`? Назовите два разных прочтения.
Если парсер вернул два дерева, означает ли это ошибку программы?
Какой дополнительный контекст помог бы выбрать одно из них?


In [ ]:
ambiguous_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> 'I' | Det N | NP PP
    VP -> V NP | VP PP
    PP -> P NP
    Det -> 'the' | 'a'
    N -> 'student' | 'telescope'
    V -> 'saw'
    P -> 'with'
""")
ambiguous_parser = ChartParser(ambiguous_grammar)
ambiguous_tokens = 'I saw the student with a telescope'.split()
ambiguous_trees = list(ambiguous_parser.parse(ambiguous_tokens)) # два разных варианта парсинга

print('Деревьев:', len(ambiguous_trees))
for number, tree in enumerate(ambiguous_trees, 1):
    vp = tree[1]
    pp_attaches_to_vp = isinstance(vp[-1], Tree) and vp[-1].label() == 'PP'
    reading = 'PP при VP: инструмент наблюдения' if pp_attaches_to_vp else 'PP внутри NP: студент с телескопом'
    print(f'\nРазбор {number}. {reading}')
    tree.pretty_print()


Деревьев: 2

Разбор 1. PP при VP: инструмент наблюдения
     S                                        
  ___|_____________                            
 |                 VP                         
 |        _________|__________                 
 |       VP                   PP              
 |    ___|___             ____|___             
 |   |       NP          |        NP          
 |   |    ___|_____      |     ___|______      
 NP  V  Det        N     P   Det         N    
 |   |   |         |     |    |          |     
 I  saw the     student with  a      telescope


Разбор 2. PP внутри NP: студент с телескопом
     S                                        
  ___|_____________                            
 |                 VP                         
 |    _____________|_____                      
 |   |                   NP                   
 |   |        ___________|____                 
 |   |       |                PP              
 |   |       |            ____|___         

<details>
<summary>После обсуждения: пояснение</summary>

Два дерева допустимы по этой грамматике: неоднозначность — свойство сочетания **входа и грамматики**.
Парсер перечисляет разрешённые структуры, а не устанавливает, что произошло в мире.
`ChartParser` здесь не ранжирует деревья по вероятности.

В правилах есть левая рекурсия `NP -> NP PP`, `VP -> VP PP`.
Нельзя просто заменить парсер на наивный `RecursiveDescentParser` и ожидать безопасного завершения.
Chart сохраняет промежуточные гипотезы вместо такого рекурсивного спуска.

</details>


### Дополнительно: зависимости связывают слова

В дереве составляющих узлами были NP и VP. В базовом представлении зависимостей
у каждого токена, кроме корня, указан главный токен и тип связи.
Ниже связи **заданы вручную для объяснения**


In [ ]:
from nltk.parse.dependencygraph import DependencyGraph

dependency_data = '\n'.join([
    'The\tDT\t2\tdet',
    'students\tNNS\t3\tnsubj',
    'read\tVBP\t0\troot',
    'the\tDT\t5\tdet',
    'books\tNNS\t3\tobj',
    '.\t.\t3\tpunct',
])
dependency_graph = DependencyGraph(dependency_data, top_relation_label='root')
print(f'{"Токен":10} {"Главный":10} Связь')
for index, node in sorted(dependency_graph.nodes.items()):
    if index:
        head = dependency_graph.nodes[node['head']]['word'] or 'ROOT'
        print(f'{node["word"]:10} {head:10} {node["rel"]}')
dependency_graph.tree().pretty_print()


Токен      Главный    Связь
The        students   det
students   read       nsubj
read       ROOT       root
the        books      det
books      read       obj
.          read       punct
      read        
  _____|_______    
 |  students books
 |     |       |   
 .    The     the 



`students` зависит от `read` как подлежащее (`nsubj`), `books` — как объект (`obj`).
Наличие связей не следует только из того, что слова стоят рядом.
В настоящем тексте парсер может ошибаться и в выборе главного слова, и в типе связи.
Для русского применимы те же идеи, но нужен подходящий языковой анализатор и схема разметки.


## 6. NER: именная группа и именованная сущность

`the old library` — NP, но не обязательно имя конкретного объекта.
`New York` может быть упоминанием географической сущности.
**Чанкинг NP** ищет грамматические группы, **NER** — границы и типы именованных упоминаний.
NER сам по себе не устанавливает уникальный объект базы знаний: для этого нужна отдельная привязка сущностей.

В NLTK `ne_chunk()` принимает английский Penn POS. Для него не переводим теги в Universal
и не удаляем заранее регистр, служебные слова или пунктуацию.


### Стоп 9

Какие сущности ожидаете в `Barack Obama visited Paris` и `Apple opened an office in London`?
Может ли слово с заглавной буквы быть обычным словом? Может ли NP не быть именованной сущностью?



In [ ]:
ner_sentences = ['Barack Obama visited Paris.', 'Apple opened an office in London.']
ner_tagged = nltk.pos_tag_sents([word_tokenize(s) for s in ner_sentences]) # выводим POS теги
ner_trees = list(nltk.ne_chunk_sents(ner_tagged, binary=False)) # по контексту понимает, где именованная сущность

for sentence, tree in zip(ner_sentences, ner_trees):
    print('\nТекст:', sentence)
    print(tree.pformat(margin=100))
    for child in tree:
        if isinstance(child, Tree):
            print(' ', child.label(), '->', ' '.join(word for word, _ in child.leaves()))



Текст: Barack Obama visited Paris.
(S (PERSON Barack/NNP) (PERSON Obama/NNP) visited/VBD (GPE Paris/NNP) ./.)
  PERSON -> Barack
  PERSON -> Obama
  GPE -> Paris

Текст: Apple opened an office in London.
(S (PERSON Apple/NNP) opened/VBD an/DT office/NN in/IN (GPE London/NNP) ./.)
  PERSON -> Apple
  GPE -> London


<details>
<summary>После обсуждения: пояснение</summary>

Сопоставляем ожидания с **реальным выводом**, а не считаем каждый результат модели правильным.
Она может ошибиться в типе или разбить полное имя на несколько упоминаний.
`binary=False` сохраняет предсказанные типы; `binary=True` свёл бы именованные сущности к NE.
Тип `GPE` обозначает геополитическую сущность в схеме этой модели.
Одинаковая строка в разных контекстах также может обозначать разные объекты.

</details>


### От упоминаний к кандидату на отношение

NER нашёл **кто/что упоминается**. Отношение требует ещё установить связь.
В учебном примере возьмём уже **вручную размеченные** PERSON и ORGANIZATION
и очень узкий шаблон `PERSON works at ORGANIZATION`.
Так отдельно проверяем извлечение связи, не смешивая его с ошибками NER.


In [ ]:
relation_example = Tree('S', [ # так задают дерево разбора
    Tree('PERSON', [('Alice', 'NNP')]),
    ('works', 'VBZ'), ('at', 'IN'),
    Tree('ORGANIZATION', [('Acme', 'NNP')]),
    ('.', '.'),
])

def employment_candidates(tree): # парсит дерево и ищет нужное поддерево
    """Ищет кандидатов PERSON works at ORGANIZATION в плоской разметке."""
    candidates = []
    parts = list(tree)
    for i in range(len(parts) - 3):
        left, verb, prep, right = parts[i:i + 4]
        if (isinstance(left, Tree) and left.label() == 'PERSON'
                and isinstance(right, Tree) and right.label() == 'ORGANIZATION'
                and isinstance(verb, tuple) and verb[0].lower() == 'works'
                and isinstance(prep, tuple) and prep[0].lower() == 'at'):
            person = ' '.join(word for word, _ in left.leaves())
            organization = ' '.join(word for word, _ in right.leaves())
            candidates.append((person, 'works_at', organization))
    return candidates

print(employment_candidates(relation_example))


[('Alice', 'works_at', 'Acme')]


Это **кандидат по текстовому шаблону**, а не установленный факт трудоустройства.
Нужно учитывать отрицание, время, цитирование и источник утверждения.
Шаблон пропустит перефразировки и не распознаёт все возможные отношения.

**Вопрос для короткой дискуссии:** если фраза встречается в «Bob claims that Alice works at Acme»,
доказано ли, что Alice действительно там работает? Что нужно сохранить вместе с извлечённой тройкой?


## 7. Мини-конвейер для итогового проекта

Соберём обработку небольшого отрывка Chesterton: предложения → токены → POS → NP и NER.
Для демонстрации берём **8 предложений**, чтобы вывод оставался читаемым.
В варианте 6 проекта требуется 300 предложений и более полный отчёт.

NP и сущности считаем отдельно: их границы и типы не обязаны совпадать.
Для NP ниже объединяем только различия регистра. Сущности сохраняем как строки исходного текста;
одинаковые строки не гарантируют идентичности реального объекта.


In [ ]:
def prepare_sentences(raw_text, limit=8):
    """Возвращает автоматический Penn POS для первых limit предложений."""
    sentences = sent_tokenize(raw_text)[:limit]
    return nltk.pos_tag_sents([word_tokenize(sentence) for sentence in sentences])

def count_noun_phrases(tagged_sentences, parser):
    """Считает поверхностные строки NP без различий регистра."""
    counts = Counter()
    for tagged in tagged_sentences:
        counts.update(phrase.lower() for phrase in noun_phrases(parser.parse(tagged)))
    return counts

def count_named_entities(tagged_sentences):
    """Считает строки предсказанных именованных упоминаний отдельно по типам."""
    counts = defaultdict(Counter)
    for tree in nltk.ne_chunk_sents(tagged_sentences, binary=False):
        for child in tree:
            if isinstance(child, Tree):
                name = ' '.join(word for word, _ in child.leaves())
                counts[child.label()][name] += 1
    return counts

project_tagged = prepare_sentences(gutenberg.raw('chesterton-thursday.txt'), limit=8)
project_np_counts = count_noun_phrases(project_tagged, np_parser)
project_entity_counts = count_named_entities(project_tagged)

print('Предложений:', len(project_tagged))
print('Частые NP:', project_np_counts.most_common(10))
for label, counts in sorted(project_entity_counts.items()):
    print(label, '| упоминаний:', sum(counts.values()), '| частые:', counts.most_common(5))


Предложений: 8
Частые NP: [('men', 3), ('the world', 3), ('the man who was thursday', 1), ('g. k. chesterton', 1), (']', 1), ('edmund clerihew bentley a cloud', 1), ('the mind', 1), ('the weather', 1), ('yea', 1), ('a sick cloud', 1)]
GPE | упоминаний: 2 | частые: [('Science', 1), ('Whistler', 1)]
PERSON | упоминаний: 6 | частые: [('Edmund Clerihew Bentley', 1), ('Yea', 1), ('Men', 1), ('Life', 1), ('Baal', 1)]



В художественном тексте заголовки, необычные имена и регистр могут отличаться от обучающих данных модели.
Неверный POS может повредить и NP, и NER: проверяем **шаг конвейера**, где возникло расхождение.


## Что запомнить

- POS, границы NP, полный синтаксический разбор и тип сущности — разные слои анализа.
- `RegexpParser` применяет заданные правила к тегам; CFG задаёт допустимые структуры, парсер ищет деревья.
- Одно IOB-представление описывает плоские непересекающиеся группы.
- Точная граница группы важна: высокая токенная accuracy не гарантирует высокий chunk F1.
- Несколько разборов могут отражать неоднозначность, отсутствие разбора — неполноту нашей грамматики.
- NER не равно выделению всех NP, привязке сущности или проверке истинности факта.


### Стоп 10

**Перед завершением — ответьте по одному предложению:**

1. На что смотрит правило `<DT>?<JJ.*>*<NN.*>+`: на буквы, POS-теги или прошлые IOB-метки?
2. Почему 80% IOB accuracy могут сочетаться с 0% chunk F1?
3. Если для новой фразы нет дерева, что проверите раньше: грамматичность языка или покрытие нашей грамматики?



<details>
<summary>После обсуждения: пояснение</summary>

1. На последовательность POS-тегов.
2. Ошибка в границе может испортить единственную целую группу при большинстве верных меток.
3. Сначала наличие терминалов в словаре и подходящих правил в нашей грамматике.


</details>


## Материалы

- [NLTK: chunking и информационное извлечение](https://www.nltk.org/book/ch07.html) — темы для повторения.
- [NLTK: грамматики и парсинг](https://www.nltk.org/book/ch08.html).
- [RegexpParser: API](https://www.nltk.org/api/nltk.chunk.regexp.html).
- [ChunkScore: оценка групп](https://www.nltk.org/api/nltk.chunk.util.html).
- [ChartParser: API](https://www.nltk.org/api/nltk.parse.chart.html).
- [Исходник PerceptronTagger](https://www.nltk.org/_modules/nltk/tag/perceptron.html).
- [Перевод тегов в pos_tag](https://www.nltk.org/_modules/nltk/tag.html).
- [WordNet: синсеты и отношения](https://www.nltk.org/howto/wordnet.html).
- [WordNet: происхождение базы](https://wordnet.princeton.edu/frequently-asked-questions).
- [Universal POS и дополнительные признаки](https://universaldependencies.org/u/pos/).
- [Adversarial examples и обучение устойчивости: исходная работа](https://arxiv.org/abs/1412.6572).

Примеры и учебные грамматики здесь составлены для занятия; они намеренно ограничены.
В старых примерах NLTK Book могут встречаться другие имена ресурсов и способы оценки:
в этом ноутбуке используются актуальные табличные ресурсы NLTK и `ChunkScore`.
